In [0]:
#Load and inspect the members table
df_members = spark.table("default.members")
df_members.printSchema()
df_members.display()

In [0]:
#Determine if any employee is current or terminated

from pyspark.sql import functions as F

df_elig = df_members.withColumn(
    "coverage_status",
    F.when(F.col("termination_date").isNull(), "Current")
    .otherwise("Terminated")
)

# df_elig labels members as current if the termination date is null, otherwise terminated by adding an additional column labled "coverage_status" with two categories
# either "Current" or "Terminated".


#display info by coverage status
df_elig.groupBy("coverage_status").count().display()

In [0]:
# Check for data quality to see if current employee is missing critical registration info

df_dq = df_elig.withColumn(
    "missing_phone", F.col("phone").isNull()
).withColumn(
    "missing_email", F.col("email").isNull()
).withColumn(
    "missing_zip", F.col("zip_code").isNull()
).withColumn(
    "missing_dob", F.col("dob").isNull()
).withColumn(
    "missing_critical_info",
    F.col("missing_phone") | F.col("missing_email") | F.col("missing_zip") | F.col("missing_dob")
)
# This code checks to if any critical info is missing but first it create a column for each critical and a cumulative column for if nay conatct column is missing info

df_dq.display()

In [0]:

# Show only members with at least one missing critical field
df_dq.filter(F.col("missing_critical_info")).select(
    "member_id", "member_name", "coverage_status",
    "missing_phone", "missing_email", "missing_zip", "missing_dob"
).display()

# Quick summary: how many active members have incomplete registration data
df_dq.filter(F.col("missing_critical_info") & (F.col("coverage_status") == "Active")) \
    .count()


In [0]:
df_visits = spark.table("default.patient_visits")

df_check = df_visits.join(df_elig, "member_id") \
    .withColumn(
        "was_eligible_at_visit",
        F.when(
            (F.col("visit_date") >= F.col("enrollment_date")) &
            (F.col("termination_date").isNull() | (F.col("visit_date") <= F.col("termination_date"))),
            True
        ).otherwise(False)
    )
# The above code joins eligible patients table we created with patient visits table
# It verifies if a member was eligible at the time of their visit, based on their enrollment and termination dates.
# If the visit date is between the enrollment date and the termination date (or if there is no termination date), the member is considered eligible.
# Otherwise, the member is considered not eligible

df_check.groupBy("was_eligible_at_visit").count().display()

#was_eligible_at_visit count
#False               1375
#True                374

In [0]:
#Here we begin investigating different conditions for why patients were not eligble for claims

In [0]:
# To begin we will isolate ineglible patients at the time of visit
df_ineligible_visits = df_check.filter(F.col("was_eligible_at_visit") == False)
# we will extract only key columns

df_ineligible_visits.select(
    "visit_id", "member_id", "member_name", "visit_date",
    "enrollment_date", "termination_date", "coverage_status"
).display()

# Breakdown: how many are "visit before enrollment" vs "visit after termination"
df_ineligible_visits.withColumn(
    "ineligibility_reason",
    F.when(F.col("visit_date") < F.col("enrollment_date"), "Visit before enrollment")
     .when(F.col("visit_date") > F.col("termination_date"), "Visit after termination")
     .otherwise("Unknown")
).groupBy("ineligibility_reason").count().display()

In [0]:
# Now we tie this inelibile patient visits to the cliams table and see if these were correctly or erronously paid

In [0]:
df_claims = spark.table("default.claims")

df_claims.columns


In [0]:
# Since pyspark 
v = df_ineligible_visits.alias("v")
c = df_claims.alias("c")

df_leakage_check = v.join(c, v.visit_id == c.visit_id, "left") \
    .select(
        "v.visit_id", "v.member_id", "v.member_name", "c.claim_status", "c.denial_reason", "c.paid_amount"
    )
# We join the claims table to the ineligble visits table to see if any of these claims were paid


In [0]:
df_leakage_check.display()

In [0]:
# We examine the reasons why cliams were denied
df_leakage_check.filter(F.col("claim_status") == "Denied").groupBy("denial_reason").count().display()

In [0]:
# we will examine the claims that were paid to inelibile visits
df_elig_paid = df_leakage_check.filter(F.col("claim_status") == "Paid")

In [0]:
df_elig_paid.filter(F.col("paid_amount").isNotNull()).agg(F.sum("paid_amount").alias("total_paid")).display()